<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [1]:
# Install spark

In [2]:
!pip install pyspark
!pip install findspark

import findspark
findspark.init()

In [3]:
# Start session

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('MLOps').getOrCreate()

25/05/30 08:01:35 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/30 08:01:37 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [18]:
# Download The search term dataset from the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [20]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

--2025-05-30 08:03:03--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv’

searchterms.csv     100%[===================>] 227.99K  --.-KB/s    in 0.007s  

2025-05-30 08:03:04 (30.0 MB/s) - ‘searchterms.csv’ saved [233457/233457]



In [6]:
# Load the csv into a spark dataframe

In [21]:
data = spark.read.csv('searchterms.csv', header=True, inferSchema=True)

In [7]:
# Print the number of rows and columns
# Take a screenshot of the code and name it as shape.jpg)

In [25]:
rows = data.count()

cols = len(data.columns)

print(f'Rows: {rows}, cols: {cols}')

Rows: 10000, cols: 4


In [24]:
# Print the top 5 rows
# Take a screenshot of the code and name it as top5rows.jpg)

In [26]:
data.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



In [9]:
# Find out the datatype of the column searchterm?
# Take a screenshot of the code and name it as datatype.jpg)

In [32]:
data.schema["searchterm"].dataType

StringType

In [10]:
# How many times was the term `gaming laptop` searched?
# Take a screenshot of the code and name it as gaminglaptop.jpg)

In [34]:
data.filter(data["searchterm"] == "gaming laptop").count()

499

In [11]:
# Print the top 5 most frequently used search terms?
# Take a screenshot of the code and name it as top5terms.jpg)

In [35]:
data.groupBy("searchterm").count().show(5)

[Stage 18:===========================================>          (81 + 10) / 100]

+-------------------+-----+
|         searchterm|count|
+-------------------+-----+
|          mobile 5g| 2301|
|ebooks data science|  410|
|      mobile 6 inch| 2312|
|     tablet 10 inch|  715|
|             laptop|  935|
+-------------------+-----+
only showing top 5 rows



In [12]:
# The pretrained sales forecasting model is available at  the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

In [13]:
# Load the sales forecast model.
# Take a screenshot of the code and name it as loadmodel.jpg)

In [42]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

import tarfile

with tarfile.open("model.tar.gz","r:gz") as tar:
    tar.extractall()

from pyspark.ml.regression import LinearRegressionModel
model = LinearRegressionModel.load('sales_prediction.model')

--2025-05-30 08:28:54--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1490 (1.5K) [application/x-tar]
Saving to: ‘model.tar.gz’

model.tar.gz        100%[===================>]   1.46K  --.-KB/s    in 0s      

2025-05-30 08:28:54 (14.7 MB/s) - ‘model.tar.gz’ saved [1490/1490]



In [45]:
# Using the sales forecast model, predict the sales for the year of 2023.
# Take a screenshot of the code and name it as forecast.jpg

from pyspark.ml.feature import VectorAssembler

def predict(year):
    
    # create a VectorAssembler to turn the year into a features column
    assembler = VectorAssembler(inputCols=["year"], outputCol="features")
    
    # input feature and data
    data = [[2023]]
    columns = ["year"]
    
    # create DataFrame with the data
    df = spark.createDataFrame(data, columns)
    
    # transform the data using VectorAssembler to create features column
    transformed_df = assembler.transform(df).select('features') # no label for prediction
    
    # use the pretrained model to make predictions
    predictions = model.transform(transformed_df)
    predictions.select('prediction').show()

TypeError: Invalid param value given for param "inputCols". Could not convert columns to list of strings

In [ ]:
predict(